# CardPilot MC-OCR: train, benchmark, export

Attach the Vietnamese Receipts MC-OCR dataset and a private Kaggle Dataset containing the CardPilot OCR source. This notebook keeps research outputs separate from the production service.

In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/vietnamese-receipts-mc-ocr-2021')
SOURCE_ROOT = Path('/kaggle/input/cardpilot-ocr-source')
WORK_ROOT = Path('/kaggle/working/cardpilot-ocr')
MODEL_VERSION = 'mcocr-cardpilot-v1'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
assert DATASET_ROOT.exists(), DATASET_ROOT
assert SOURCE_ROOT.exists(), SOURCE_ROOT

## 1. Inspect data and build the fixed benchmark manifest
Keep this manifest unchanged between experiments. Do not use its cases for training.

In [ ]:
images = [p for p in DATASET_ROOT.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
print(f'{len(images):,} images')
print(*images[:10], sep='\n')

In [ ]:
import shutil
import subprocess
import sys

annotations = next(SOURCE_ROOT.rglob('mcocr_train_df.csv'))
prepare = next(SOURCE_ROOT.rglob('prepare_mcocr.py'))
manifest = WORK_ROOT / 'benchmark.jsonl'
subprocess.run([
    sys.executable, str(prepare),
    '--dataset-root', str(DATASET_ROOT),
    '--annotations', str(annotations),
    '--output', str(manifest),
], check=True)

## 2. Train or fine-tune
MC-OCR is four models, not one model. Train only the component under experiment, then point the paths below at its best checkpoint. Keep the other three checkpoints at the frozen baseline. The legacy upstream commands live under `mc_ocr/rotation_corrector`, `mc_ocr/text_classifier/vietocr`, and `mc_ocr/key_info_extraction/PICK`.

In [ ]:
# Download and normalize the four upstream baseline checkpoints. Kaggle Internet must be on.
if subprocess.run([sys.executable, '-c', 'import gdown']).returncode != 0:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)

checkpoint_root = WORK_ROOT / 'checkpoints'
downloader = next(SOURCE_ROOT.rglob('download_models.py'))
subprocess.run([
    sys.executable, str(downloader),
    '--root', str(SOURCE_ROOT),
    '--model-dir', str(checkpoint_root),
    '--model-version', 'mc-ocr-top1-upstream',
], check=True)

# Override one of these after training when evaluating a new component.
DETECTOR = checkpoint_root / 'detector/ch_ppocr_server_v2.0_det_infer'
ROTATION = checkpoint_root / 'rotation/model.pth'
RECOGNITION = checkpoint_root / 'recognition/model.pth'
KIE = checkpoint_root / 'kie/model.pth'

# Example for a component-specific training command:
# subprocess.run(['bash', str(SOURCE_ROOT / 'mc_ocr/key_info_extraction/PICK/dist_train.sh')], check=True)

## 3. Export the immutable artifact
The output directory is the only interface consumed by `cardpilot-ocr-service`.

In [ ]:
checkpoints = {
    'detector': DETECTOR,
    'rotation': ROTATION,
    'recognition': RECOGNITION,
    'kie': KIE,
}
missing = [f'{name}: {path}' for name, path in checkpoints.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing OCR checkpoints:\n' + '\n'.join(missing))

exporter = next(SOURCE_ROOT.rglob('export_artifact.py'))
artifact = WORK_ROOT / 'artifacts' / MODEL_VERSION
subprocess.run([
    sys.executable, str(exporter),
    '--detector', str(DETECTOR),
    '--rotation', str(ROTATION),
    '--recognition', str(RECOGNITION),
    '--kie', str(KIE),
    '--model-version', MODEL_VERSION,
    '--output', str(artifact),
], check=True)
shutil.make_archive(str(artifact), 'zip', artifact)

## 4. Benchmark the exported artifact
Run the engine in-process against the fixed manifest. Install the service runtime dependencies in a compatible Kaggle image first. Record field accuracy plus model-stage latency. Do not compare Kaggle latency with production CPU latency.

In [ ]:
service_root = next(p.parent for p in SOURCE_ROOT.rglob('cardpilot_service') if p.is_dir())
benchmark_runner = next(SOURCE_ROOT.rglob('in_process.py'))
report = WORK_ROOT / f'{MODEL_VERSION}-report.json'
subprocess.run([
    sys.executable, str(benchmark_runner),
    '--manifest', str(manifest),
    '--service-root', str(service_root),
    '--model-dir', str(artifact),
    '--device', 'cuda',
    '--output', str(report),
], check=True)